### LLM Initiation

In [12]:
import langchain
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv("./.env")

google_api_key = os.getenv("GOOGLE_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

google_llm = ChatGoogleGenerativeAI(
    temperature=0, 
    model="gemini-2.5-flash", 
    api_key=google_api_key,
    max_tokens=200
)

openai_llm = ChatOpenAI(
    temperature=0, 
    model="gpt-4o", 
    api_key=openai_api_key
)


##### Normal Text prompt

In [3]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser


prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("human", "{input}")
])

chain = prompt | google_llm | StrOutputParser()

chain.invoke({"input": "What is 2 + 2?"})

'2 + 2 = 4'

### Base64 image encoded upload

In [ ]:
from langchain_core.messages import HumanMessage
import base64

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


# image_path = "images/taj_mahal.jpg"
image_path = "images/taj-mahal-vance.webp"
base64_image = encode_image(image_path)

message = HumanMessage(content=[
    {
        "type": "text",
        "text": "Detect  objects in this image and list them"
    },
    {
        "type": "image_url",
        "image_url": {"url": f"data:image/webp;base64,{base64_image}"}
    }
])

res = google_llm.invoke([message])
print(res)

### Base64 method - Automatic MIME type detection

In [ ]:
from langchain_core.messages import HumanMessage
import base64
import mimetypes

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


image_path = "images/taj_mahal.jpg"
# image_path = "images/taj-mahal-vance.webp"
base64_image = encode_image(image_path)

mime_type, _ = mimetypes.guess_type(image_path)
if not mime_type:
    mime_type = "image/jpeg"

message = HumanMessage(content=[
    {
        "type": "text",
        "text": "Detect  objects in this image and list them"
    },
    {
        "type": "image_url",
        "image_url": {"url": f"data:{mime_type};base64,{base64_image}"}
    }
])

res = openai_llm.invoke([message])
print(res)

### Using native google-generativeai SDK to upload file and reference

In [ ]:
import google.generativeai as genai
import mimetypes

genai.configure(api_key=google_api_key)

def upload_to_gemini(path):
    mime_type, _ = mimetypes.guess_type(path)
    if not mime_type:
        mime_type = "image/jpeg"
    file = genai.upload_file(path, mime_type=mime_type)
    print(f"Uploaded file {file.display_name} as {file.uri}")
    return file


image_path = "images/taj_mahal.jpg"
uploaded_file = upload_to_gemini(image_path)

model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content([
    "Identify objects in this image and list them",
    uploaded_file
])

print(response)


# FOLLOWING LANGCHAIN METHOD WON'T WORK CAUSE ChatGoogleGenerativeAI does not recognize the following internal URI
# "https://generativelanguage.googleapis.com/v1beta/files/3uveporc0wwr"



AIMessage(content='The image contains the following objects:\n\n1. Lake\n2. Mountains\n3. Trees\n4. Sky\n5. Clouds\n6. Person (near the left side)\n7. Rocks\n8. Reflections in the water', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 440, 'total_tokens': 487, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a0e9480a2f', 'id': 'chatcmpl-D2z8IpWqS55oUo1ptoyRqzp1drraT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--5c39de7d-f8d3-421d-a573-c116af996a61-0', usage_metadata={'input_tokens': 440, 'output_tokens': 47, 'total_tokens': 487, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [26]:
from langchain_core.messages import HumanMessage
import mimetypes

message = HumanMessage(content=[
    {
        "type": "text",
        "text": "Identify objects in this image and list them"
    },
    {
        "type": "image_url",
        "image_url": {"url": "https://img.freepik.com/free-photo/beautiful-lake-mountains_395237-44.jpg"}
    }
])


openai_llm.invoke([message])



AIMessage(content='The image contains the following objects:\n\n1. Lake\n2. Mountains\n3. Trees\n4. Sky\n5. Clouds\n6. Person (near the left side)\n7. Rocks\n8. Reflections in the water', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 440, 'total_tokens': 487, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a0e9480a2f', 'id': 'chatcmpl-D2zAFNEV0IwdwKAcrQCPbbmebByaQ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--a0a999ac-6cc2-4f43-8318-a8d18dff446b-0', usage_metadata={'input_tokens': 440, 'output_tokens': 47, 'total_tokens': 487, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})